In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    build_metadata,
    parse_filename,
    audit_data_folder,
    get_label,
    load_recording,
    load_currents,
    inspect_recordings,
    scan_signal_sanity,
    scan_corpus,
    summarize_anomalies,
    build_bearings_table,
    CHANNEL_CATALOG
)


In [2]:
metadata = build_metadata(refresh=True)
print("shape:", metadata.shape)
display(metadata.head())

shape: (2557, 7)


,bearing,speed_rpm,torque_Nm,force_N,rep,filepath,label
0,K001,900,0.7,1000,1,/home/marwaneze/Projects/Personal/Motor_PM/dat...,healthy
1,K001,900,0.7,1000,10,/home/marwaneze/Projects/Personal/Motor_PM/dat...,healthy
2,K001,900,0.7,1000,11,/home/marwaneze/Projects/Personal/Motor_PM/dat...,healthy
3,K001,900,0.7,1000,12,/home/marwaneze/Projects/Personal/Motor_PM/dat...,healthy
4,K001,900,0.7,1000,13,/home/marwaneze/Projects/Personal/Motor_PM/dat...,healthy


In [15]:
print("bearings:", sorted(metadata["bearing"].unique()))
print("label counts:\n", metadata["label"].value_counts().to_string())
print()
print("bearings per class:\n", metadata.groupby("label")["bearing"].nunique().to_string())
print()
print("operating conditions:\n", metadata[["speed_rpm", "torque_Nm", "force_N"]].drop_duplicates().to_string())

bearings: ['K001', 'K002', 'K003', 'K004', 'K005', 'K006', 'KA01', 'KA03', 'KA04', 'KA05', 'KA06', 'KA07', 'KA08', 'KA09', 'KA15', 'KA16', 'KA22', 'KA30', 'KB23', 'KB24', 'KB27', 'KI01', 'KI03', 'KI04', 'KI05', 'KI07', 'KI08', 'KI14', 'KI16', 'KI17', 'KI18', 'KI21']
label counts:
 label
damaged    2077
healthy     480

bearings per class:
 label
damaged    26
healthy     6

operating conditions:
     speed_rpm  torque_Nm  force_N
0         900        0.7     1000
20       1500        0.1     1000
40       1500        0.7      400
60       1500        0.7     1000


In [16]:
samples = [
    "N15_M07_F10_K001_3",
    "N09_M07_F10_KA01_11",
    "N15_M01_F10_KI21_20",
    "N15_M07_F04_KB23_1",
]
for stem in samples:
    print(f"{stem:28s} -> {parse_filename(stem)}")

N15_M07_F10_K001_3           -> {'bearing': 'K001', 'speed_rpm': 1500, 'torque_Nm': 0.7, 'force_N': 1000, 'rep': 3}
N09_M07_F10_KA01_11          -> {'bearing': 'KA01', 'speed_rpm': 900, 'torque_Nm': 0.7, 'force_N': 1000, 'rep': 11}
N15_M01_F10_KI21_20          -> {'bearing': 'KI21', 'speed_rpm': 1500, 'torque_Nm': 0.1, 'force_N': 1000, 'rep': 20}
N15_M07_F04_KB23_1           -> {'bearing': 'KB23', 'speed_rpm': 1500, 'torque_Nm': 0.7, 'force_N': 400, 'rep': 1}


In [17]:
healthy = (metadata["label"] == "healthy").sum()
damaged = (metadata["label"] == "damaged").sum()
print(f"healthy: {healthy} | damaged: {damaged} | ratio 1 : {damaged / healthy:.2f}")
print("-> imbalance drives metric choice (F1 / PR-AUC, not accuracy).")

healthy: 480 | damaged: 2077 | ratio 1 : 4.33
-> imbalance drives metric choice (F1 / PR-AUC, not accuracy).


In [18]:
audit = audit_data_folder(verify_loads=False)
print("bearings found:", len(audit["bearings"]))
print(f"present: {audit['present_files']} / expected: {audit['expected_files']}")
print("missing:", audit["missing"] or "none")
print("load failures:", [f for f in audit["load_failures"]] or "none")

[audit] MISSING 3 expected recordings
bearings found: 32
present: 2557 / expected: 2560
missing: [('KA05', 'N15_M07_F10', 16), ('KA08', 'N15_M01_F10', 2), ('KA08', 'N15_M07_F10', 11)]
load failures: none


In [19]:
display(pd.DataFrame(CHANNEL_CATALOG).T)


,rate,samples,role,unit
vibration_1,64000,256001,fault_primary,g
phase_current_1,64000,256001,fault_secondary,A
phase_current_2,64000,256001,fault_secondary,A
speed,4000,16001,condition,rpm
torque,4000,16001,condition,Nm
force,4000,16001,condition,N
temp_2_bearing_module,1,5,condition,degC


In [20]:
path = metadata[(metadata["bearing"] == "K001") & (metadata["rep"] == 1)]["filepath"].iloc[0]
print("file:", path)
report = inspect_recordings(path)
display(pd.DataFrame(report).T)

file: /home/marwaneze/Projects/Personal/Motor_PM/data/raw/K001/N09_M07_F10_K001_1.mat


,shape,dtype,samples,min,max,expected_samples,matches_catalog
force,"(16008,)",float64,16008,791.45957,1291.337499,16001,False
phase_current_1,"(256823,)",float64,256823,-2.853623,2.877041,256001,False
phase_current_2,"(256823,)",float64,256823,-2.769591,2.750305,256001,False
speed,"(16008,)",float64,16008,899.413253,900.140777,16001,False
temp_2_bearing_module,"(5,)",float64,5,46.710205,47.009277,5,True
torque,"(16008,)",float64,16008,1.002617,1.471853,16001,False
vibration_1,"(256823,)",float64,256823,-4.879761,4.690552,256001,False


In [21]:
from collections import Counter

lengths = Counter()
for b in ["K001", "K003", "KA03", "KI21"]:
    rec = metadata[(metadata["bearing"] == b) & (metadata["rep"] == 1)].iloc[0]
    s = load_recording(rec["filepath"])
    L = s["vibration_1"].size
    lengths[L] += 1
    print(f"{b:5s} vibration_1 = {L:>7} samples  ({'catalog exact match' if L == CHANNEL_CATALOG['vibration_1']['samples'] else 'differs from nominal'})")

print()
print("-> recording lengths are NOT constant; ~22% of the corpus is longer than nominal.")
print("-> match_catalog=False during exploration is EXPECTED; cleaning trims to a fixed length.")

K001  vibration_1 =  256823 samples  (differs from nominal)
K003  vibration_1 =  256000 samples  (differs from nominal)
KA03  vibration_1 =  256001 samples  (catalog exact match)
KI21  vibration_1 =  256003 samples  (differs from nominal)

-> recording lengths are NOT constant; ~22% of the corpus is longer than nominal.
-> match_catalog=False during exploration is EXPECTED; cleaning trims to a fixed length.


In [22]:
paths = [
    metadata[(metadata["bearing"] == "K001") & (metadata["rep"] == 1)]["filepath"].iloc[0],  # healthy
    metadata[(metadata["bearing"] == "KA01") & (metadata["rep"] == 1)]["filepath"].iloc[0],  # artificial
    metadata[(metadata["bearing"] == "KB23") & (metadata["rep"] == 1)]["filepath"].iloc[0],  # real (fatigue)
]
for p in paths:
    print("\n==", p)
    display(pd.DataFrame(scan_signal_sanity(p)).T[["samples", "nan", "inf", "flat", "min", "max", "within_limits"]])


== /home/marwaneze/Projects/Personal/Motor_PM/data/raw/K001/N09_M07_F10_K001_1.mat


,samples,nan,inf,flat,min,max,within_limits
force,16008,0,0,False,791.45957,1291.337499,True
phase_current_1,256823,0,0,False,-2.853623,2.877041,True
phase_current_2,256823,0,0,False,-2.769591,2.750305,True
speed,16008,0,0,False,899.413253,900.140777,True
temp_2_bearing_module,5,0,0,False,46.710205,47.009277,True
torque,16008,0,0,False,1.002617,1.471853,True
vibration_1,256823,0,0,False,-4.879761,4.690552,True



== /home/marwaneze/Projects/Personal/Motor_PM/data/raw/KA01/N09_M07_F10_KA01_1.mat


,samples,nan,inf,flat,min,max,within_limits
force,16001,0,0,False,868.058691,1228.471288,True
phase_current_1,256001,0,0,False,-3.003088,2.928011,True
phase_current_2,256001,0,0,False,-3.015486,2.706912,True
speed,16001,0,0,False,898.78652,900.742217,True
temp_2_bearing_module,5,0,0,False,46.115112,46.472168,True
torque,16001,0,0,False,1.122331,1.374956,True
vibration_1,256001,0,0,False,-2.145386,2.404785,True



== /home/marwaneze/Projects/Personal/Motor_PM/data/raw/KB23/N09_M07_F10_KB23_1.mat


,samples,nan,inf,flat,min,max,within_limits
force,16001,0,0,False,666.032324,1390.214452,True
phase_current_1,256001,0,0,False,-2.921123,2.813673,True
phase_current_2,256001,0,0,False,-3.202146,2.910792,True
speed,16001,0,0,False,898.9098,900.001004,True
temp_2_bearing_module,5,0,0,False,53.967285,54.208374,True
torque,16001,0,0,False,1.03599,1.411705,True
vibration_1,256001,0,0,False,-3.665161,3.564453,True


In [23]:
scan = scan_corpus(max_files=50)
print(f"scanned {scan['filepath'].nunique()} recordings")
display(summarize_anomalies(scan))

scanned 50 recordings


,files,nan_files,inf_files,flat_files,out_of_limits,min_value,max_value
channel,,,,,,,
force,50,0,0,0,0,129.228125,1370.378026
phase_current_1,50,0,0,0,0,-3.057502,3.094696
phase_current_2,50,0,0,0,0,-3.044415,3.105717
speed,50,0,0,0,0,898.914224,1500.463161
temp_2_bearing_module,50,0,0,0,0,44.140625,47.009277
torque,50,0,0,0,0,0.389224,1.512252
vibration_1,50,0,0,0,0,-5.419922,5.184937


In [24]:
full_scan = scan_corpus()
display(summarize_anomalies(full_scan))
full_scan.to_csv("../data/processed/anomaly_report.csv", index=False)

  scanned 400/2557
  scanned 800/2557
  scanned 1200/2557
  scanned 1600/2557
  scanned 2000/2557
  scanned 2400/2557


,files,nan_files,inf_files,flat_files,out_of_limits,min_value,max_value
channel,,,,,,,
force,2557,0,0,0,1,-0.471582,1561.418065
phase_current_1,2557,0,0,0,0,-3.260004,3.144977
phase_current_2,2557,0,0,0,0,-3.765570,4.377897
speed,2557,0,0,0,0,679.619290,1516.690914
temp_2_bearing_module,2557,0,0,0,1,-1.147461,78.372192
torque,2557,0,0,0,1,-0.191534,1.578656
vibration_1,2557,0,0,0,11,-9.573364,10.928345


In [ ]:
bearings_table = build_bearings_table(metadata)
display(bearings_table)

display(bearings_table.groupby("damage_type").agg(
    bearings=("bearing", "count"),
    recordings=("n_recordings", "sum"),
))

saved: /home/marwaneze/Projects/Personal/Motor_PM/data/processed/bearings.csv


,bearing,damage_type,n_recordings,conditions,note
0,K001,healthy,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
1,K002,healthy,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
2,K003,healthy,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
3,K004,healthy,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
4,K005,healthy,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
5,K006,healthy,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
6,KA01,artificial,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
7,KA03,artificial,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
8,KA04,artificial,80,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",
9,KA05,artificial,79,"[N09_M07_F10, N15_M01_F10, N15_M07_F04, N15_M0...",


,bearings,recordings
damage_type,,
artificial,12,957
combination,3,240
healthy,6,480
real,11,880



-> KA (artificial) is the bulk of the damaged corpus.
-> 'real' KI bearings are the true generalization target. if few KI recordings exist, train/test on real damage needs care.
